# PneumoType: what kind of pneumonia is this X-ray?

**User:** a doctor at a busy children's clinic who has to decide fast whether a child with a cough needs antibiotics.
**Input → output:** one chest X-ray → **Normal / Bacterial pneumonia / Viral pneumonia**, a confidence score, a heatmap, and **"Unsure, refer"** when the model isn't confident.

Why the *type* matters: bacterial pneumonia needs antibiotics; viral usually doesn't. The dangerous mistake is calling a **bacterial** case **viral or normal** (the child misses antibiotics).

**Everything is trained from scratch:** our own convolutional neural network, random starting weights, no pretrained model.

> **This project is a prototype and is not medical advice.**

**Setup on Kaggle:** New Notebook → *Add Input* → search **"Chest X-Ray Images (Pneumonia)"** (Paul Mooney) → add it. Settings → Accelerator → **GPU T4**. Turn Internet **on** (needed for `pip install`). Then Run All.

In [ ]:
!pip install -q gradio

## 1. Settings
`MODE="3class"` predicts normal / bacterial / viral. `MODE="binary"` predicts just normal vs pneumonia. Our model is a CNN we designed ourselves and train **from scratch** (no pretrained weights).

In [ ]:
import os, re, glob, random, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
IMG_SIZE = 224
BATCH = 32
EPOCHS = 25         # small CNN: ~30-60 s per epoch on a T4
LR = 1e-3
MODE = "3class"     # "3class" (normal/bacterial/viral) or "binary" (normal/pneumonia)
DATA_ROOT = "/kaggle/input"
CLASSES = ["normal", "bacterial", "viral"] if MODE == "3class" else ["normal", "pneumonia"]
K = len(CLASSES)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 2. Load file list and labels
Labels come from the folder (NORMAL / PNEUMONIA) and, for pneumonia, from the filename (`person12_bacteria_45.jpeg` or `person3_virus_7.jpeg`).
We also pull out a **patient ID** so the same child never appears in both training and test (otherwise the score is inflated).

In [ ]:
paths = [p for p in glob.glob(f"{DATA_ROOT}/**/*.jpeg", recursive=True) if "__MACOSX" not in p]

def label_of(p):
    folder = os.path.basename(os.path.dirname(p)).upper()
    name = os.path.basename(p).lower()
    if folder == "NORMAL":
        return "normal"
    if folder == "PNEUMONIA":
        if MODE == "binary": return "pneumonia"
        if "bacteria" in name: return "bacterial"
        if "virus" in name: return "viral"
    return None

def patient_of(p):
    name = os.path.basename(p)
    m = re.match(r"(person\d+)_", name, re.I)
    if m: return m.group(1).lower()
    m = re.match(r"(.*?IM-\d+)-\d+", name, re.I)
    if m: return m.group(1).upper()
    return os.path.splitext(name)[0]

df = pd.DataFrame({"path": paths})
df["label"] = df.path.map(label_of)
df = df.dropna(subset=["label"])
df["fname"] = df.path.map(os.path.basename)
df = df.drop_duplicates(subset=["label", "fname"])   # the Kaggle copy has a duplicated nested folder
df["patient"] = df.path.map(patient_of)
df["y"] = df.label.map({c: i for i, c in enumerate(CLASSES)})

print(len(df), "images,", df.patient.nunique(), "patients")
print(df.label.value_counts())
assert len(df) > 1000, "Images not found: check the dataset is attached and DATA_ROOT is right"

## 3. Split by patient: 70% train / 15% validation / 15% test
The test set is locked away until the very end. Validation is used to pick the best epoch and the thresholds.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

def group_split(d, n_splits, seed):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    a, b = next(sgkf.split(d, d.y, groups=d.patient))
    return d.iloc[a].reset_index(drop=True), d.iloc[b].reset_index(drop=True)

trainval, test_df = group_split(df, 7, SEED)          # ~15% test
train_df, val_df  = group_split(trainval, 6, SEED)    # ~15% val

assert not set(train_df.patient) & set(test_df.patient)
assert not set(val_df.patient) & set(test_df.patient)
for n, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(n, len(d), d.label.value_counts().to_dict())

## 4. Baseline: always guess the most common class
Our model has to beat this. Accuracy alone can look good here, so we also report **macro recall** (the average of per-class recall), which a lazy model can't game.

In [ ]:
from sklearn.metrics import (accuracy_score, recall_score, precision_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)

def report(name, y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    rec = recall_score(y_true, y_pred, labels=list(range(K)), average=None, zero_division=0)
    yt_p = (y_true > 0).astype(int); yp_p = (y_pred > 0).astype(int)
    row = {"model": name, "accuracy": accuracy_score(y_true, y_pred), "macro_recall": rec.mean()}
    for c, r in zip(CLASSES, rec): row[f"recall_{c}"] = r
    row["pneumonia_recall(any type)"] = recall_score(yt_p, yp_p, zero_division=0)
    row["false_alarms(normal called pneumonia)"] = int(((yt_p == 0) & (yp_p == 1)).sum())
    row["missed_pneumonia(called normal)"] = int(((yt_p == 1) & (yp_p == 0)).sum())
    row[f"DANGER {CLASSES[1]} missed"] = int(((y_true == 1) & (y_pred != 1)).sum())
    return row

results = []
majority = train_df.y.mode()[0]
results.append(report(f"Baseline: always '{CLASSES[majority]}'", test_df.y, [majority] * len(test_df)))
pd.DataFrame(results)

## 5. Images → tensors
Light augmentation for training only (small rotations/zoom/brightness). No left-right flips: the heart is on one side, so flipping makes unrealistic X-rays.

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
])
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor()])

def to_model_input(x):
    # x: [B,1,H,W] in [0,1]
    return (x - 0.5) / 0.25                # simple centering; 1 grayscale channel

class XRays(Dataset):
    def __init__(self, d, tf): self.d, self.tf = d.reset_index(drop=True), tf
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        r = self.d.iloc[i]
        img = Image.open(r.path).convert("L")
        return self.tf(img), int(r.y)

nw = 2
train_dl = DataLoader(XRays(train_df, train_tf), batch_size=BATCH, shuffle=True, num_workers=nw)
val_dl   = DataLoader(XRays(val_df, eval_tf), batch_size=BATCH, num_workers=nw)
test_dl  = DataLoader(XRays(test_df, eval_tf), batch_size=BATCH, num_workers=nw)

## 6. Model
Our own CNN, trained from random weights. Five blocks of *convolution → batch norm → ReLU → max-pool*, doubling the number of filters each block (32 → 512), then global average pooling, dropout and a final linear layer.
- **Convolutions** learn small patterns (edges, then textures, then shapes like hazy patches in the lungs).
- **Max-pooling** shrinks the image so later layers see bigger regions.
- **Batch norm** keeps training stable; **dropout** reduces overfitting.
About 4.7M parameters, small enough to train in minutes on a free GPU.

In [ ]:
def block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.MaxPool2d(2))

class OurCNN(nn.Module):
    def __init__(self):
        super().__init__()
        chans = [1, 32, 64, 128, 256, 512]
        self.features = nn.Sequential(*[block(a, b) for a, b in zip(chans[:-1], chans[1:])])

class PneumoType(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = OurCNN()
        self.drop = nn.Dropout(0.3)
        self.head = nn.Linear(512, K)
    def forward(self, x):
        f = self.backbone.features(to_model_input(x))
        f = F.adaptive_avg_pool2d(F.relu(f), 1).flatten(1)
        return self.head(self.drop(f))
    def embed(self, x):
        f = self.backbone.features(to_model_input(x))
        return F.adaptive_avg_pool2d(F.relu(f), 1).flatten(1)

model = PneumoType().to(device)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M params")

## 7. A second baseline: logistic regression on raw pixels
A simple, non-deep model: shrink each X-ray to 32×32 pixels and fit a logistic regression. If our CNN can't clearly beat this, it isn't learning much.

In [ ]:
from sklearn.linear_model import LogisticRegression

from sklearn.preprocessing import StandardScaler
def pixels(d):
    return np.stack([np.asarray(Image.open(p).convert("L").resize((32, 32)), dtype=np.float32).ravel() / 255 for p in d.path])

sc = StandardScaler().fit(pixels(train_df))
lr = LogisticRegression(max_iter=3000, class_weight="balanced", C=0.1).fit(sc.transform(pixels(train_df)), train_df.y)
results.append(report("Baseline: logistic regression on 32x32 pixels", test_df.y, lr.predict(sc.transform(pixels(test_df)))))
pd.DataFrame(results)

## 8. Train our CNN from scratch
Class weights make the rarer classes count more, so the model doesn't just ignore it. We keep the epoch with the best **macro recall on validation** (never on test).

In [ ]:
counts = train_df.y.value_counts().sort_index().values
class_w = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_w)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

@torch.no_grad()
def predict_probs(dl):
    model.eval(); P, Y = [], []
    for x, y in dl:
        with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
            P.append(F.softmax(model(x.to(device)).float(), 1).cpu().numpy())
        Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)

best, history = -1, []
for ep in range(EPOCHS):
    model.train(); t0 = time.time(); tot = 0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item() * len(y)
    sched.step()
    pv, yv = predict_probs(val_dl)
    mr = recall_score(yv, pv.argmax(1), average="macro")
    history.append((ep + 1, tot / len(train_df), mr))
    print(f"epoch {ep+1}: train loss {tot/len(train_df):.3f} | val macro recall {mr:.3f} | {time.time()-t0:.0f}s")
    if mr > best:
        best = mr; torch.save(model.state_dict(), "best.pt")

model.load_state_dict(torch.load("best.pt"))
print("best val macro recall:", round(best, 3))

## 9. Results on the held-out test set

In [ ]:
p_test, y_test = predict_probs(test_dl)
pred_test = p_test.argmax(1)
results.append(report("Our CNN (from scratch)", y_test, pred_test))
display(pd.DataFrame(results).round(3))

ConfusionMatrixDisplay(confusion_matrix(y_test, pred_test, labels=list(range(K))),
                       display_labels=CLASSES).plot(cmap="Blues")
plt.title("Test set confusion matrix"); plt.show()

## 10. Safety tuning: make the dangerous mistake rarer
We add a small bonus to the score of class 1 (**bacterial** in 3-class mode, **pneumonia** in binary mode), choosing the smallest bonus that keeps its **recall ≥ 95% on validation**, then check what it costs on test.
We also add an **"Unsure, refer"** band: if the top probability is below a cutoff, the tool doesn't decide.

In [ ]:
def adjusted_pred(p, bonus):
    q = p.copy(); q[:, 1] = q[:, 1] + bonus
    return q.argmax(1)

p_val, y_val = predict_probs(val_dl)
TARGET = 0.95
chosen = 0.0
for b in np.arange(0, 0.6, 0.01):
    if recall_score(y_val, adjusted_pred(p_val, b), labels=[1], average=None)[0] >= TARGET:
        chosen = b; break
print(f"{CLASSES[1]} bonus chosen on validation:", round(chosen, 2))

results.append(report(f"Our CNN + safety bonus {chosen:.2f}", y_test, adjusted_pred(p_test, chosen)))
display(pd.DataFrame(results).round(3))

# Unsure band
for cut in [0.5, 0.6, 0.7, 0.8]:
    conf = p_test.max(1); keep = conf >= cut
    acc = accuracy_score(y_test[keep], adjusted_pred(p_test, chosen)[keep]) if keep.any() else float("nan")
    print(f"cutoff {cut}: decides on {keep.mean():.0%} of X-rays, accuracy on those {acc:.3f}, refers {(~keep).sum()}")

## 11. Where does it fail?
The most confident **wrong** answers on the test set. Look at them and write down *why* you think the model got each one wrong (bad image quality, mild disease, devices or labels on the image, etc.).

In [ ]:
wrong = np.where(pred_test != y_test)[0]
wrong = wrong[np.argsort(-p_test[wrong].max(1))][:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, i in zip(axes.flat, wrong):
    ax.imshow(Image.open(test_df.path[i]).convert("L"), cmap="gray")
    ax.set_title(f"true {CLASSES[y_test[i]]} | pred {CLASSES[pred_test[i]]} ({p_test[i].max():.2f})", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 12. Robustness check: what if the clinic's X-ray machine is different?
This dataset has no age or sex labels (all children aged 1–5 from one hospital), so we can't do a demographic fairness split. Instead we test whether performance survives a **different-looking X-ray** (lower contrast, blur, lower resolution), which is a real issue when a tool moves to a new clinic. Say this limitation in your write-up.

In [ ]:
def shifted_tf(kind):
    ops = [T.Resize((IMG_SIZE, IMG_SIZE))]
    if kind == "low contrast": ops.append(T.ColorJitter(contrast=(0.5, 0.5)))
    if kind == "blur": ops.append(T.GaussianBlur(5, sigma=1.5))
    if kind == "low resolution": ops = [T.Resize((96, 96)), T.Resize((IMG_SIZE, IMG_SIZE))]
    ops.append(T.ToTensor()); return T.Compose(ops)

rob = []
for kind in ["original", "low contrast", "blur", "low resolution"]:
    dl = DataLoader(XRays(test_df, shifted_tf(kind) if kind != "original" else eval_tf), batch_size=BATCH, num_workers=nw)
    p, y = predict_probs(dl)
    r = report(kind, y, adjusted_pred(p, chosen)); rob.append(r)
pd.DataFrame(rob)[["model", "accuracy", "macro_recall", "pneumonia_recall(any type)"]].round(3)

## 13. Heatmap: what is the model looking at? (Grad-CAM)

In [ ]:
def gradcam(img_pil, target=None):
    model.eval()
    x = eval_tf(img_pil.convert("L")).unsqueeze(0).to(device).requires_grad_(False)
    acts = {}
    def fwd_hook(m, i, o):
        acts["a"] = o; o.register_hook(lambda g: acts.__setitem__("g", g))
    h = model.backbone.features.register_forward_hook(fwd_hook)
    logits = model(x); h.remove()
    probs = F.softmax(logits, 1)[0].detach().cpu().numpy()
    probs_adj = probs.copy(); probs_adj[1] += chosen
    cls = int(probs_adj.argmax()) if target is None else target
    model.zero_grad(); logits[0, cls].backward()
    w = acts["g"].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * acts["a"]).sum(1)).squeeze().detach().cpu().numpy()
    cam = np.array(Image.fromarray(cam).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return probs, cls, cam

fig, axes = plt.subplots(1, K, figsize=(5 * K, 5))
for ax, c in zip(axes, range(K)):
    i = test_df.index[test_df.y == c][0]
    img = Image.open(test_df.path[i]).convert("L")
    probs, cls, cam = gradcam(img)
    ax.imshow(img.resize((IMG_SIZE, IMG_SIZE)), cmap="gray"); ax.imshow(cam, cmap="jet", alpha=0.35)
    ax.set_title(f"true {CLASSES[c]} | pred {CLASSES[cls]}"); ax.axis("off")
plt.show()

## 14. Demo app (Gradio)
Run this cell, click the public link it prints, upload an X-ray. Record your demo video from this.

In [ ]:
import gradio as gr
UNSURE_CUTOFF = 0.6   # pick from the table in step 10

def diagnose(img):
    if img is None: return "Upload an X-ray", None
    pil = Image.fromarray(img) if not isinstance(img, Image.Image) else img
    probs, cls, cam = gradcam(pil)
    if probs.max() < UNSURE_CUTOFF:
        verdict = "UNSURE: refer to a doctor / radiologist"
    else:
        verdict = ({0: "Normal", 1: "Bacterial pneumonia: likely needs antibiotics, see a doctor now",
                    2: "Viral pneumonia: see a doctor"} if MODE == "3class" else
                   {0: "Normal", 1: "Pneumonia likely: see a doctor now"})[cls]
    text = verdict + "\n\n" + "\n".join(f"{c}: {p:.0%}" for c, p in zip(CLASSES, probs))
    text += "\n\nPrototype only. Not medical advice."
    base = np.array(pil.convert("RGB").resize((IMG_SIZE, IMG_SIZE))) / 255.0
    heat = plt.cm.jet(cam)[..., :3]
    overlay = (0.65 * base + 0.35 * heat)
    return text, (overlay * 255).astype(np.uint8)

gr.Interface(fn=diagnose, inputs=gr.Image(label="Chest X-ray"),
             outputs=[gr.Textbox(label="Result"), gr.Image(label="Where the model looked")],
             title="PneumoType: what kind of pneumonia?",
             description="Prototype for a children's clinic. Not medical advice.").launch(share=True)

## 15. Numbers for your write-up
Copy the final table and fill in the template: baseline score vs model score, the dangerous-mistake count before and after the safety bonus, the unsure-band trade-off, the robustness drop, and 1–2 failure examples with your explanation.

**Credits to list:** Kermany et al. 2018 chest X-ray dataset (Kaggle: paultimothymooney/chest-xray-pneumonia, CC BY 4.0), PyTorch, scikit-learn, Gradio, and any AI assistants you used. No pretrained models.